In [1]:
import chess
import numpy as np
from typing import Optional
import torch
from torchrl.envs import EnvBase
from torchrl.data import (
    Composite,
    Unbounded,
    Categorical,
    Bounded,
    ReplayBuffer,
    LazyTensorStorage,
)
from tensordict import TensorDict
import torch.nn as nn
from tensordict.nn import TensorDictModule
from torchrl.objectives import DQNLoss

In [2]:
class ChessEnvironment:
    """
    Raw chess environment before TorchRL wrapping.
    python-chess handles all move validation, check detection,
    checkmate, stalemate etc. We just need to handle the
    state representation and reward signal.
    """
 
    def __init__(self):
        self.board = chess.Board()
        self.move_history = []
 
    def reset(self):
        self.board = chess.Board()
        self.move_history = []
        return self._get_observation()
 
    def step(self, action: int):
        """
        Action is an integer index into the list of legal moves.
        Returns observation, reward, done, info.
        """
        legal_moves = list(self.board.legal_moves)
 
        if action >= len(legal_moves):
            # Illegal action chosen — penalize and end episode
            return self._get_observation(), -10.0, True, {"reason": "illegal_move"}
 
        move = legal_moves[action]
        self.board.push(move)
        self.move_history.append(move)
 
        done = self.board.is_game_over()
        reward = self._compute_reward(done)
 
        return self._get_observation(), reward, done, {}
 
    def _get_observation(self):
        """
        Encode the board as a tensor.
        12 channels x 8 x 8:
          - 6 piece types for white (pawn, knight, bishop, rook, queen, king)
          - 6 piece types for black
        Each cell is 1.0 if that piece occupies that square, 0.0 otherwise.
        This is the standard representation used in AlphaZero.
        """
        obs = np.zeros((12, 8, 8), dtype=np.float32)
 
        piece_map = {
            (chess.PAWN,   chess.WHITE): 0,
            (chess.KNIGHT, chess.WHITE): 1,
            (chess.BISHOP, chess.WHITE): 2,
            (chess.ROOK,   chess.WHITE): 3,
            (chess.QUEEN,  chess.WHITE): 4,
            (chess.KING,   chess.WHITE): 5,
            (chess.PAWN,   chess.BLACK): 6,
            (chess.KNIGHT, chess.BLACK): 7,
            (chess.BISHOP, chess.BLACK): 8,
            (chess.ROOK,   chess.BLACK): 9,
            (chess.QUEEN,  chess.BLACK): 10,
            (chess.KING,   chess.BLACK): 11,
        }
 
        for square, piece in self.board.piece_map().items():
            channel = piece_map[(piece.piece_type, piece.color)]
            row = square // 8
            col = square % 8
            obs[channel, row, col] = 1.0
 
        return obs
 
    def _compute_reward(self, done: bool):
        """
        Reward signal is a key design decision in RL.
        This is a simple version — you can make it much richer.
        """
        if not done:
            return 0.0  # no reward for intermediate moves
 
        if self.board.is_checkmate():
            # Whoever just moved won
            # board.turn flips after each move, so if it's
            # white's turn now, black just moved and won
            return 1.0 if self.board.turn == chess.WHITE else -1.0
 
        # Stalemate, draw by repetition, insufficient material etc.
        return 0.0
 
    def get_legal_move_count(self):
        return len(list(self.board.legal_moves))
 
    def render(self):
        print(self.board)
        print()

In [3]:
class TorchRLChessEnv(EnvBase):
 
    MAX_MOVES = 218
 
    def __init__(self, device="cpu"):
        super().__init__(device=device, batch_size=[])
 
        self.chess_env = ChessEnvironment()
 
        self.obs_shape = (12, 8, 8)
        self.action_dim = self.MAX_MOVES
 
        # -------------------------------------------------
        # Required specs (torchrl >= 0.5)
        # observation_spec covers all non-reward/done outputs
        # -------------------------------------------------
        self.observation_spec = Composite(
            observation=Unbounded(
                shape=torch.Size([12, 8, 8]),
                dtype=torch.float32,
                device=device,
            ),
            legal_move_count=Bounded(
                low=0,
                high=self.MAX_MOVES,
                shape=torch.Size([]),
                dtype=torch.int64,
                device=device,
            ),
            shape=torch.Size([]),
            device=device,
        )
 
        self.action_spec = Composite(
            action=Categorical(
                n=self.MAX_MOVES,
                shape=torch.Size([]),
                dtype=torch.int64,
                device=device,
            ),
            shape=torch.Size([]),
            device=device,
        )
 
        # reward shape must be [..., 1] to satisfy torchrl conventions
        self.reward_spec = Composite(
            reward=Unbounded(
                shape=torch.Size([1]),
                dtype=torch.float32,
                device=device,
            ),
            shape=torch.Size([]),
            device=device,
        )
 
        # done / terminated must match reward shape [..., 1]
        self.done_spec = Composite(
            done=Categorical(
                n=2,
                shape=torch.Size([1]),
                dtype=torch.bool,
                device=device,
            ),
            terminated=Categorical(
                n=2,
                shape=torch.Size([1]),
                dtype=torch.bool,
                device=device,
            ),
            shape=torch.Size([]),
            device=device,
        )
 
    # -------------------------
    # RESET
    # -------------------------
    def _reset(self, tensordict=None):
        obs = self.chess_env.reset()
 
        return TensorDict(
            {
                "observation": torch.tensor(obs, dtype=torch.float32),
                "legal_move_count": torch.tensor(
                    self.chess_env.get_legal_move_count(),
                    dtype=torch.int64,
                ),
            },
            batch_size=[],
        )
 
    # -------------------------
    # STEP
    # -------------------------
    def _step(self, tensordict):
        action = tensordict["action"].item()
 
        obs, reward, done, info = self.chess_env.step(action)
 
        # torchrl wraps _step output under the "next" key automatically.
        # reward, done, terminated must have shape [..., 1] to match specs.
        return TensorDict(
            {
                "observation": torch.tensor(obs, dtype=torch.float32),
                "reward": torch.tensor([reward], dtype=torch.float32),
                "done": torch.tensor([done], dtype=torch.bool),
                "terminated": torch.tensor([done], dtype=torch.bool),
                "legal_move_count": torch.tensor(
                    self.chess_env.get_legal_move_count(),
                    dtype=torch.int64,
                ),
            },
            batch_size=[],
        )
 
    # -------------------------
    # SEED
    # -------------------------
    def _set_seed(self, seed: Optional[int]):
        if seed is not None:
            torch.manual_seed(seed)

In [4]:
class ChessNet(nn.Module):
    """
    Convolutional Q-network for chess.
    Input: 12 x 8 x 8 board representation
    Output: Q-value for each of the 218 possible actions
    
    Architecture loosely inspired by AlphaZero but much
    simpler — a full AlphaZero would use residual blocks
    and separate policy/value heads.
    """
 
    def __init__(self, num_actions=218):
        super().__init__()
 
        self.conv_layers = nn.Sequential(
            # First conv: capture local piece interactions
            nn.Conv2d(12, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
 
            # Second conv: wider spatial patterns
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
 
            # Third conv: high-level positional features
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )
 
        # After three convolutions the spatial dims are
        # still 8x8 because we used padding=1 throughout
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_actions)
        )
 
    def forward(self, observation):
        x = self.conv_layers(observation)
        return self.fc_layers(x)
 
 
def build_chess_actor(env):
    net = ChessNet(num_actions=env.MAX_MOVES)
 
    actor = TensorDictModule(
        net,
        in_keys=["observation"],
        out_keys=["action_value"]
    )
 
    return actor

In [5]:
class LegalMoveMaskedActor(nn.Module):
    """
    Wraps the Q-network and masks illegal moves before
    selecting an action. Without masking the agent will
    frequently try illegal moves and learn very slowly.
    """
 
    def __init__(self, q_network, chess_env):
        super().__init__()
        self.q_network = q_network
        self.chess_env = chess_env
 
    def forward(self, tensordict):
        obs = tensordict["observation"]
        q_values = self.q_network(obs)
 
        # Build a mask: 1.0 for legal moves, -inf for illegal
        legal_count = tensordict["legal_move_count"].item()
        mask = torch.full_like(q_values, float("-inf"))
        mask[:legal_count] = 0.0
 
        # Add mask to Q-values so illegal moves
        # are never selected by argmax
        masked_q = q_values + mask
 
        action = masked_q.argmax(dim=-1)
        tensordict["action"] = action
        tensordict["action_value"] = masked_q
        return tensordict
 
 
 
def train_chess_agent(num_episodes=10000):
 
    env = TorchRLChessEnv()
    actor = build_chess_actor(env)
    masked_actor = LegalMoveMaskedActor(actor, env.chess_env)
 
    buffer = ReplayBuffer(
        storage=LazyTensorStorage(max_size=50000)
    )
 
    # action_space must be a string or TensorSpec, not an integer.
    # gamma is no longer a DQNLoss constructor argument in 0.12 —
    # set it via make_value_estimator() instead.
    loss_fn = DQNLoss(
        value_network=actor,
        action_space="categorical",
    )
    loss_fn.make_value_estimator(gamma=0.99)
 
    optimizer = torch.optim.Adam(actor.parameters(), lr=1e-4)
 
    epsilon = 1.0
    epsilon_decay = 0.9995
    epsilon_min = 0.05
 
    for episode in range(num_episodes):
        tensordict = env.reset()
        episode_reward = 0.0
        done = False
 
        while not done:
 
            # Epsilon-greedy exploration
            if torch.rand(1).item() < epsilon:
                # Random legal move
                legal_count = tensordict["legal_move_count"].item()
                tensordict["action"] = torch.randint(0, legal_count, ()).to(torch.int64)
            else:
                # Best legal move according to Q-network
                with torch.no_grad():
                    tensordict = masked_actor(tensordict)
 
            # env.step() returns a TensorDict where next-state data lives
            # under the "next" nested key (handled by EnvBase automatically).
            next_tensordict = env.step(tensordict)
            done = next_tensordict["next", "done"].squeeze(-1).item()
            reward = next_tensordict["next", "reward"].item()
            episode_reward += reward
 
            # Buffer expects a batched TensorDict; unsqueeze adds batch dim.
            buffer.extend(next_tensordict.unsqueeze(0))
 
            # The new current obs is the next state
            tensordict = next_tensordict["next"].clone()
 
            # Train when buffer is big enough
            if len(buffer) >= 1000:
                sample = buffer.sample(batch_size=64)
                loss = loss_fn(sample)
 
                optimizer.zero_grad()
                loss["loss"].backward()
 
                # Gradient clipping helps stabilize chess training
                torch.nn.utils.clip_grad_norm_(
                    actor.parameters(), max_norm=1.0
                )
 
                optimizer.step()
 
        # Decay exploration
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
 
        if episode % 100 == 0:
            print(f"Episode {episode} | "
                  f"Reward: {episode_reward:.2f} | "
                  f"Epsilon: {epsilon:.3f} | "
                  f"Buffer: {len(buffer)}")
 
    return actor

In [6]:
if __name__ == "__main__":
 
    env = TorchRLChessEnv()
 
    print("Testing environment...")
    td = env.reset()
 
    print(f"Initial observation shape: {td['observation'].shape}")
    print(f"Legal moves available: {td['legal_move_count'].item()}")
 
    # ---- take random move ----
    td_step = TensorDict(
        {"action": torch.tensor(0, dtype=torch.int64)},
        batch_size=[],
    )
 
    result = env.step(td_step)
 
    result = result["next"]
 
    print(f"Reward after first move: {result['reward'].item()}")
    print(f"Done: {bool(result['done'])}")
 
    print("\nStarting training...")
    trained_actor = train_chess_agent(num_episodes=10000)
 
    print("\nWatching a game...")
 
    td = env.reset()
    done = False
    move_count = 0
 
    while not done and move_count < 100:
 
        env.chess_env.render()
 
        with torch.no_grad():
 
            # Actor must output ACTION, not TensorDict
            action = trained_actor(td["observation"])
 
            # If actor returns TensorDict, extract action safely
            if isinstance(action, dict) or hasattr(action, "keys"):
                action = action["action"]
 
            action = torch.as_tensor(action, dtype=torch.int64).view(())
 
        td = env.step(
            TensorDict({"action": action}, batch_size=[])
        )
 
        td = td["next"]
 
        done = bool(td["done"])
        move_count += 1

Testing environment...
Initial observation shape: torch.Size([12, 8, 8])
Legal moves available: 20
Reward after first move: 0.0
Done: False

Starting training...


KeyError: "value key 'chosen_action_value' not found in value network out_keys ['action_value']"